# India Rate Shock Lab
## How Do RBI Monetary Policy Changes Propagate Through Indian Equity Sectors?

**Author:** Avani Aravind  
**Date:** September 2026  
**Methodology:** Event Study + OLS Rate-Sensitivity Regression

---

### Research Question
When the RBI changes its repo rate, which equity sectors react most strongly — and is that reaction immediate or persistent?

We examine **23 RBI policy change events** from 2014 to 2025 across **6 NSE sector indices** using:
1. **Event-study abnormal returns** (market-model adjusted)
2. **Cumulative Abnormal Returns (CARs)** at horizons [0], [0,+1], [0,+4], [0,+19]
3. **Rate Sensitivity Scores (β₂)** from a two-factor OLS regression

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import statsmodels.api as sm
import plotly.io as pio
pio.renderers.default = 'notebook'

from src.rbi_events import get_events_df
from src.data_loader import download_prices, load_or_generate_mock, compute_returns
from src.event_study import compute_abnormal_returns, compute_car, compute_rate_sensitivity
from src.visualizations import (
    plot_rbi_timeline, plot_acar_heatmap, plot_car_fan,
    plot_rate_sensitivity, plot_rolling_beta, plot_sector_cumulative_returns
)

pd.set_option('display.float_format', '{:.4f}'.format)
print('Libraries loaded ✓')

---
## 1. Data & Policy Event Calendar

In [ ]:
events = get_events_df()
print(f'Total rate-change events: {len(events)}')
print(f'  Hikes: {len(events[events.cycle=="Hike"])}  |  Cuts: {len(events[events.cycle=="Cut"])}')
events[['date','change_bps','rate_after','stance','cycle']]

In [ ]:
# Download or use synthetic fallback
try:
    prices = download_prices(force_refresh=False)
    if prices is None or prices.empty or len(prices) < 100:
        raise ValueError('insufficient data')
    print('Live data loaded ✓')
except Exception as e:
    print(f'Yahoo Finance unavailable ({e}). Using synthetic data.')
    prices = load_or_generate_mock()

returns = compute_returns(prices)
print(f'Price data: {prices.shape[0]} rows × {prices.shape[1]} sectors')
print(f'Date range: {prices.index[0].date()} → {prices.index[-1].date()}')
prices.tail()

In [ ]:
fig = plot_rbi_timeline(events)
fig.show()

---
## 2. Descriptive Statistics

In [ ]:
ann_ret  = returns.mean()  * 252 * 100
ann_vol  = returns.std()   * np.sqrt(252) * 100
sharpe   = ann_ret / ann_vol

desc = pd.DataFrame({'Ann. Return (%)': ann_ret, 'Ann. Vol (%)': ann_vol, 'Sharpe': sharpe})
print('=== Annualised Sector Statistics ===')
print(desc.round(2).to_string())
desc.round(2)

In [ ]:
corr = returns.corr()
fig_corr, ax = plt.subplots(figsize=(8,6))
im = ax.imshow(corr, cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.index)
plt.colorbar(im, ax=ax)
ax.set_title('Daily Return Correlation Matrix', fontsize=14)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('../output/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Event Study

### Methodology
For each event *t*:
- **Estimation window**: trading days [−125, −6] (≈ 100 days)
- **Event window**: [−5, +20] trading days around the announcement

**Market model**: $r_{i,\tau} = \hat{\alpha}_i + \hat{\beta}_i \cdot r_{m,\tau} + \epsilon_{i,\tau}$

**Abnormal Return**: $AR_{i,\tau} = r_{i,\tau} - (\hat{\alpha}_i + \hat{\beta}_i \cdot r_{m,\tau})$

**CAR**: $CAR_i(a,b) = \sum_{\tau=a}^{b} AR_{i,\tau}$

In [ ]:
result   = compute_abnormal_returns(returns, market_col='Nifty50')
ar_df    = result['ar_df']
betas_df = result['betas_df']

print(f'AR observations: {len(ar_df)}')
print(f'Events studied:  {ar_df["event_date"].nunique()}')
ar_df.head(10)

In [ ]:
car_df = compute_car(ar_df)
print('CAR DataFrame shape:', car_df.shape)
car_df.head()

In [ ]:
# === ACAR Table (All events) ===
horizon_cols = [c for c in car_df.columns if c.startswith('CAR[')]
acar_table = car_df.groupby('sector')[horizon_cols].mean() * 100
acar_table.columns = [c.replace('CAR','ACAR') for c in acar_table.columns]
print('=== Average CAR (%) — All Events ===')
print(acar_table.round(3).to_string())
acar_table.round(3)

In [ ]:
# Hike vs Cut split
for cycle in ['Hike','Cut']:
    sub = car_df[car_df['cycle']==cycle]
    tab = sub.groupby('sector')[horizon_cols].mean() * 100
    print(f'\n=== ACAR (%) — {cycle} Events ({len(sub.event_date.unique())} events) ===')
    print(tab.round(3).to_string())

In [ ]:
# Interactive heatmap
fig = plot_acar_heatmap(car_df, cycle=None)
fig.show()

In [ ]:
# CAR fan charts for Banking and IT (most interesting contrast)
for sec in ['Bank','IT','Realty']:
    for cyc in ['Hike','Cut']:
        fig = plot_car_fan(ar_df, sector=sec, cycle=cyc)
        fig.show()

---
## 4. Rate Sensitivity Regression

**Model:**
$$r_{sector,t} = \alpha + \beta_1 \cdot r_{Nifty50,t} + \beta_2 \cdot \Delta Rate_t + \epsilon_t$$

- $\beta_2$ = **Rate Sensitivity Score** — the sector return attributable to a 100 bps rate change, after controlling for the broad market move.
- Standard errors: Newey-West HAC (5 lags).

In [ ]:
rate_df = compute_rate_sensitivity(returns, events)
print('=== Rate Sensitivity Scores ===')
display_cols = ['alpha','beta_market','beta_rate','t_beta_rate','p_beta_rate','r_squared','n_obs']
print(rate_df[display_cols].round(4).to_string())
rate_df[display_cols]

In [ ]:
fig = plot_rate_sensitivity(rate_df)
fig.show()

---
## 5. Rolling Beta Analysis

In [ ]:
for sec in ['Bank','IT','Realty']:
    fig = plot_rolling_beta(returns, sector=sec, window=63)
    fig.show()

---
## 6. Key Findings & Interpretation

*(Fill in after running the cells with live data.)*

### Summary of Results

| Sector | Rate Sensitivity β₂ | Interpretation |
|--------|---------------------|----------------|
| **Banking** | High negative (hike cycle) | NIM compression fears; immediate sell-off |
| **Realty**  | Most negative | High leverage → rate-sensitive debt costs |
| **IT**      | Near-zero / positive | Revenue in USD; domestic rates matter less |
| **FMCG**    | Low negative | Defensive; rate insensitive |
| **Auto**    | Moderate negative | Consumer finance costs |
| **Pharma**  | Low | Export-driven; currency matters more |

### Limitations
- Event-study assumes the market model is stable across the estimation window.
- Indian markets also respond to global factors (Fed policy, FII flows) which are not controlled.
- Yahoo Finance NSE data may have survivorship / delisting gaps for older indices.
- Small sample of rate-change events limits statistical power for sub-samples.

In [ ]:
# Export key tables to Excel for research note
from pathlib import Path
out_dir = Path('../output')
out_dir.mkdir(exist_ok=True)

with pd.ExcelWriter(out_dir / 'india_rate_shock_results.xlsx', engine='openpyxl') as writer:
    events.to_excel(writer, sheet_name='RBI_Events', index=False)
    acar_table.to_excel(writer, sheet_name='ACAR_AllEvents')
    car_df[car_df['cycle']=='Hike'].groupby('sector')[horizon_cols].mean().mul(100).round(4).to_excel(
        writer, sheet_name='ACAR_HikeCycle')
    car_df[car_df['cycle']=='Cut'].groupby('sector')[horizon_cols].mean().mul(100).round(4).to_excel(
        writer, sheet_name='ACAR_CutCycle')
    rate_df.reset_index().to_excel(writer, sheet_name='RateSensitivity', index=False)

print('Results exported to output/india_rate_shock_results.xlsx ✓')